<a href="https://colab.research.google.com/github/amina-nasrin/Graph-Neural-Network/blob/main/Task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch==2.0.1+cu117 torchvision==0.15.2+cu117 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu117

In [ ]:
!pip install torch_geometric

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
import networkx as nx

In [ ]:
!pip install torch_geometric

In [ ]:
dataset = torch_geometric.datasets.Planetoid(root='/tmp/Cora', name='Cora', split='public')

Processing...
Done!


In [ ]:
g = dataset[0]
print(f'Number of nodes: {g.num_nodes}')
print(f'Number of edges: {g.num_edges}')
print(f'Number of features: {dataset.num_node_features}')
print(f'Number of classes: {dataset.num_classes}')
print(f'Number of training nodes: {g.train_mask.sum()}')
print(f'Number of validation nodes: {g.val_mask.sum()}')
print(f'Number of test nodes: {g.test_mask.sum()}')
print(f'Has self loops?: {g.has_self_loops()}')
print(f'Is directed?: {g.is_directed()}')


Number of nodes: 2708
Number of edges: 10556
Number of features: 1433
Number of classes: 7
Number of training nodes: 140
Number of validation nodes: 500
Number of test nodes: 1000
Has self loops?: False
Is directed?: False


In [ ]:
import torch.nn as nn

class GCN(nn.Module):
  def __init__(self, in_dimension, hidden_dimension, num_classes):
    super(GCN, self).__init__()
    ##
    self.conv1 = nn.Linear(in_dimension, hidden_dimension)
    self.conv2 = nn.Linear(hidden_dimension, hidden_dimension)
    self.conv3 = nn.Linear(hidden_dimension, num_classes)
    ## 3-layer GCN model
    ##===
  def forward(self, g):
    h, edge_index = g.x, g.edge_index
    ##
    ##=== Apply F.relu to 1st and 2nd layer.
    h, edge_index = g.x, g.edge_index
    # Apply the first layer and ReLU
    h = self.conv1(h)
    h = F.relu(h)

        # Apply the second layer and ReLU
    h = self.conv2(h)
    h = F.relu(h)

    h = self.conv3(h)
    ##===
    return F.log_softmax(h, dim=1)


def train(g, model):
  optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
  best_val_acc = 0
  best_test_acc = 0
  model.train()
  for epoch in range(1, 11):
    out = model(g)
    loss = F.cross_entropy(out[g.train_mask], g.y[g.train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.eval()
    pred = model(g).argmax(dim=1)
    train_acc = (pred[g.train_mask] == g.y[g.train_mask]).float().mean()
    val_acc = (pred[g.val_mask] == g.y[g.val_mask]).float().mean()

    test_acc = (pred[g.test_mask] == g.y[g.test_mask]).float().mean()
    if best_val_acc < val_acc:
      best_val_acc = val_acc
      best_test_acc = test_acc
    if epoch % 1 == 0:
      print('In epoch {}, loss: {:.3f}, val acc: {:.3f} (best {:.3f}), test acc: {:.3f} (best {:.3f})'.format(epoch, loss, val_acc, best_val_acc, test_acc, best_test_acc))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
g = g.to(device)

model = GCN(dataset.num_node_features, 64, dataset.num_classes).to(device)
train(g, model)



In epoch 1, loss: 1.951, val acc: 0.216 (best 0.216), test acc: 0.192 (best 0.192)
In epoch 2, loss: 1.898, val acc: 0.272 (best 0.272), test acc: 0.269 (best 0.269)
In epoch 3, loss: 1.805, val acc: 0.450 (best 0.450), test acc: 0.448 (best 0.448)
In epoch 4, loss: 1.640, val acc: 0.520 (best 0.520), test acc: 0.504 (best 0.504)
In epoch 5, loss: 1.387, val acc: 0.534 (best 0.534), test acc: 0.528 (best 0.528)
In epoch 6, loss: 1.059, val acc: 0.536 (best 0.536), test acc: 0.533 (best 0.533)
In epoch 7, loss: 0.703, val acc: 0.534 (best 0.536), test acc: 0.528 (best 0.533)
In epoch 8, loss: 0.402, val acc: 0.534 (best 0.536), test acc: 0.527 (best 0.533)
In epoch 9, loss: 0.210, val acc: 0.540 (best 0.540), test acc: 0.526 (best 0.526)
In epoch 10, loss: 0.110, val acc: 0.530 (best 0.540), test acc: 0.515 (best 0.526)
